In [2]:
import re
import pandas as pd

NEGATION_WORDS = {"not", "no", "never", "n't", "cannot", "cant", "without"}
DEFAULT_STOP_WORDS = {
    "a", "an", "the", "is", "are", "was", "were", "be", "been", "being",
    "of", "in", "on", "at", "to", "for", "and", "or", "but", "with",
    "this", "that", "these", "those", "it", "its", "as", "so", "very",
    "i", "my", "me", "we", "our", "you", "your",
} - NEGATION_WORDS

def clean_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text: str) -> list:
    return text.split()

def remove_stop_words(tokens: list, stop_words: set = None) -> list:
    stop_words = stop_words if stop_words is not None else DEFAULT_STOP_WORDS
    return [t for t in tokens if t not in stop_words]

def preprocess(text: str, remove_stops: bool = True) -> str:
    cleaned = clean_text(text)
    tokens = tokenize(cleaned)
    if remove_stops:
        tokens = remove_stop_words(tokens)
    return " ".join(tokens)

In [3]:
df = pd.read_csv('feedback.csv')
df.head()

,feedback,sentiment,category
0,Payment keeps failing every time I checkout,negative,payment
1,The payment gateway failed after entering the OTP,negative,payment
2,I was charged twice for the same order,negative,payment
3,Payment went through smoothly this time,positive,payment
4,"Refund was processed within a day, thank you",positive,payment


In [4]:
sample = '   APP is VERY slow!!!   '
print(clean_text(sample))

app is very slow


In [5]:
tokens = tokenize(clean_text(sample))
print(tokens)

['app', 'is', 'very', 'slow']


In [6]:
negated = 'The application is not good'
tokens = tokenize(clean_text(negated))
print('Before:', tokens)
print('After :', remove_stop_words(tokens))

Before: ['the', 'application', 'is', 'not', 'good']
After : ['application', 'not', 'good']


In [7]:
df['clean_feedback'] = df['feedback'].apply(preprocess)
df[['feedback', 'clean_feedback']].head(10)

,feedback,clean_feedback
0,Payment keeps failing every time I checkout,payment keeps failing every time checkout
1,The payment gateway failed after entering the OTP,payment gateway failed after entering otp
2,I was charged twice for the same order,charged twice same order
3,Payment went through smoothly this time,payment went through smoothly time
4,"Refund was processed within a day, thank you",refund processed within day thank
5,The application is very slow and payment keeps...,application slow payment keeps failing
6,App is extremely slow when loading the dashboard,app extremely slow when loading dashboard
7,The application freezes frequently during chec...,application freezes frequently during checkout
8,Pages take forever to load on my phone,pages take forever load phone
9,The app crashed twice today,app crashed twice today


In [8]:
df.to_csv('feedback_clean.csv', index=False)
print('Saved feedback_clean.csv')

Saved feedback_clean.csv
